# Model Evaluation & Analysis

This notebook provides comprehensive evaluation of the trained ML surrogate model including:
- Prediction accuracy metrics
- Residual analysis
- Feature importance
- Predictions vs Actual scatter plot

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('.'))))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from physics.parameter_sampler import sample_parameters
from physics.dispersion_solver import solve_phase_velocity

print("Libraries imported successfully!")

## 1. Load Model and Data

In [ ]:
# Load trained model
with open('data/raw/model.pkl', 'rb') as f:
    model = pickle.load(f)

# Load analytical results
X_data = pd.read_csv('data/raw/analytical_results.csv').values

print(f"Model loaded: {type(model).__name__}")
print(f"Data shape: {X_data.shape}")

## 2. Regenerate Test Data with Known Targets

In [ ]:
# Generate fresh test data
X_test_raw = sample_parameters(n_samples=1000)
y_test = np.array([solve_phase_velocity(k, H, a, e) for k, H, a, e in X_test_raw])

# Remove NaN values
mask = ~np.isnan(y_test)
X_test_raw = X_test_raw[mask]
y_test = y_test[mask]

# Scale features
scaler = StandardScaler()
X_train_all = sample_parameters(n_samples=6000)
y_train_all = np.array([solve_phase_velocity(*row) for row in X_train_all])
mask_train = ~np.isnan(y_train_all)
X_train_all = X_train_all[mask_train]
y_train_all = y_train_all[mask_train]

scaler.fit(X_train_all)
X_test_scaled = scaler.transform(X_test_raw)

print(f"Test set size: {len(X_test_scaled)}")
print(f"y_test range: [{y_test.min():.4f}, {y_test.max():.4f}]")

## 3. Model Predictions

In [ ]:
# Generate predictions
y_pred = model.predict(X_test_scaled)

print(f"Predictions range: [{y_pred.min():.4f}, {y_pred.max():.4f}]")
print(f"First 10 predictions: {y_pred[:10]}")
print(f"First 10 actual values: {y_test[:10]}")

## 4. Evaluation Metrics

In [ ]:
# Calculate metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print("="*60)
print("MODEL EVALUATION METRICS")
print("="*60)
print(f"Mean Absolute Error (MAE):     {mae:.6f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.6f}")
print(f"R² Score:                      {r2:.6f}")
print(f"Mean Absolute % Error (MAPE):  {mape:.2f}%")
print("="*60)

# Interpretation
if r2 > 0.9:
    print("✓ EXCELLENT: R² > 0.9 indicates very good model performance")
elif r2 > 0.7:
    print("✓ GOOD: R² > 0.7 indicates good model performance")
else:
    print("⚠ MODERATE: R² < 0.7, model may need improvement")

## 5. Predictions vs Actual (Scatter Plot)

In [ ]:
plt.figure(figsize=(10, 8))
plt.scatter(y_test, y_pred, alpha=0.6, s=30, edgecolors='k', linewidth=0.5)

# Add perfect prediction line
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')

plt.xlabel('Analytical Phase Velocity (c)', fontsize=12)
plt.ylabel('ML Predicted Phase Velocity', fontsize=12)
plt.title(f'Model Accuracy: R² = {r2:.4f}', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/plots/predictions_vs_actual.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Plot saved to results/plots/predictions_vs_actual.png")

## 6. Residual Analysis

In [ ]:
residuals = y_test - y_pred

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Residuals vs Predicted
axes[0, 0].scatter(y_pred, residuals, alpha=0.6, s=30)
axes[0, 0].axhline(y=0, color='r', linestyle='--', lw=2)
axes[0, 0].set_xlabel('Predicted Values')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Predicted')
axes[0, 0].grid(True, alpha=0.3)

# Residuals Distribution
axes[0, 1].hist(residuals, bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Residuals')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title(f'Residuals Distribution (mean={residuals.mean():.6f})')
axes[0, 1].grid(True, alpha=0.3)

# Q-Q Plot (Normal probability plot)
from scipy import stats
stats.probplot(residuals, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot')
axes[1, 0].grid(True, alpha=0.3)

# Errors by Input Parameter
errors = np.abs(residuals)
axes[1, 1].scatter(X_test_raw[:, 0], errors, alpha=0.6, s=30)  # k vs error
axes[1, 1].set_xlabel('Wave Number (k)')
axes[1, 1].set_ylabel('Absolute Error')
axes[1, 1].set_title('Error vs Wave Number')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/plots/residual_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Residual analysis plot saved")
print(f"Mean residual: {residuals.mean():.6f}")
print(f"Std of residuals: {residuals.std():.6f}")

## 7. Feature Importance

In [ ]:
# Get feature importance from the model
if hasattr(model, 'feature_importances_'):
    feature_names = ['Wave Number (k)', 'Layer Thickness (H)', 'Heterogeneity (α)', 'Damping (η)']
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1]
    
    plt.figure(figsize=(10, 6))
    plt.bar(range(len(importances)), importances[indices], align='center')
    plt.xticks(range(len(importances)), [feature_names[i] for i in indices], rotation=45, ha='right')
    plt.ylabel('Importance')
    plt.title('Feature Importance in ML Surrogate Model')
    plt.tight_layout()
    plt.savefig('results/plots/feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("Feature Importance Ranking:")
    for i, idx in enumerate(indices):
        print(f"{i+1}. {feature_names[idx]}: {importances[idx]:.4f}")
else:
    print("Feature importance not available for this model type")

## 8. Error Distribution by Parameter Ranges

In [ ]:
errors = np.abs(residuals)
rel_errors = (errors / y_test) * 100

print("Error Statistics by Parameter Ranges:")
print("="*70)

# By k (wave number)
k_vals = X_test_raw[:, 0]
print(f"\nWave Number (k) range: [{k_vals.min():.3f}, {k_vals.max():.3f}]")
k_mask_low = k_vals < np.percentile(k_vals, 50)
print(f"  Low k:  MAE = {errors[k_mask_low].mean():.6f}, MAPE = {rel_errors[k_mask_low].mean():.2f}%")
print(f"  High k: MAE = {errors[~k_mask_low].mean():.6f}, MAPE = {rel_errors[~k_mask_low].mean():.2f}%")

# By H (layer thickness)
h_vals = X_test_raw[:, 1]
print(f"\nLayer Thickness (H) range: [{h_vals.min():.3f}, {h_vals.max():.3f}]")
h_mask_low = h_vals < np.percentile(h_vals, 50)
print(f"  Low H:  MAE = {errors[h_mask_low].mean():.6f}, MAPE = {rel_errors[h_mask_low].mean():.2f}%")
print(f"  High H: MAE = {errors[~h_mask_low].mean():.6f}, MAPE = {rel_errors[~h_mask_low].mean():.2f}%")

print("="*70)

## 9. Summary & Conclusion

In [ ]:
print("\n" + "="*70)
print("MODEL EVALUATION SUMMARY")
print("="*70)
print(f"\n📊 Accuracy Metrics:")
print(f"   • R² Score: {r2:.4f} (Explains {r2*100:.2f}% of variance)")
print(f"   • MAE: {mae:.6f}")
print(f"   • RMSE: {rmse:.6f}")
print(f"   • MAPE: {mape:.2f}%")

print(f"\n🎯 Model Status:")
if r2 > 0.95:
    print(f"   ✓ EXCELLENT - Ready for production use")
elif r2 > 0.85:
    print(f"   ✓ VERY GOOD - Suitable for most applications")
elif r2 > 0.7:
    print(f"   ✓ GOOD - Can be used with caution")
else:
    print(f"   ⚠ FAIR - Consider retraining or feature engineering")

print(f"\n📈 Next Steps:")
print(f"   1. Review residual plots for patterns")
print(f"   2. Check feature importance rankings")
print(f"   3. Consider ensemble methods if needed")
print(f"   4. Deploy for inverse design optimization")
print("\n" + "="*70)